# Proyecto de Computación Evolutiva · lectura verificable

Este cuaderno recorre los resultados de la **reconstrucción nueva**, no las salidas históricas.
Las celdas de lectura se ejecutaron secuencialmente al generar este archivo.
El entrenamiento y la optimización se ejecutan mediante `run.py`; no se afirma que este cuaderno los haya repetido.

Para reconstruir todo, ejecutar desde la raíz del proyecto:
```python
import subprocess, sys
subprocess.run([sys.executable, 'run.py', '--stage', 'all'], check=True)
```
La instantánea de DBAASP debe estar completa. Instalar antes las dependencias de `requirements.txt`.


In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'run.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'run.py').exists(), 'Abrir desde la raíz del proyecto o desde notebooks/'
print('Raíz:', ROOT)
def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))


Raíz: D:\Computación Evolutiva\Proyecto_CE_Peptidos


## 1. Problema y procedencia
Se minimiza log10 MIC y se maximiza log10 HC50 predichas. El dominio tiene 21 posiciones y hasta dos cambios. Las concentraciones generadas por el modelo no son nuevas mediciones.

In [2]:
raw = read_json('data/raw/manifest.json')
print(json.dumps({k: raw[k] for k in ['status', 'started_utc', 'finished_utc', 'filters', 'ids', 'scope']}, ensure_ascii=False, indent=2))
print(json.dumps(read_json('data/processed/resumen_datos.json'), ensure_ascii=False, indent=2))


{
  "status": "complete",
  "started_utc": "2026-10-17T17:12:41.942905+00:00",
  "finished_utc": "2026-10-17T17:27:25.990498+00:00",
  "filters": {
    "sequenceLength.value": "10-30",
    "complexity.value": "Monomer",
    "unusualAminoAcid.value": "without_modification",
    "targetSpecies.value": "Escherichia coli ATCC 25922"
  },
  "ids": 4308,
  "scope": "Todos los IDs devueltos por el filtro declarado. HC50 se extrae de esta misma cohorte; no reproduce el conjunto historico de 181/153 registros."
}
{
  "cohort_api_ids": 4308,
  "measurements_audited": 51867,
  "accepted_measurements": {
    "MIC": 2069,
    "HC50": 202
  },
  "chemical_entities": {
    "MIC": 1473,
    "HC50": 199
  },
  "exclusion_reasons": {
    "otro_objetivo_biologico": 35378,
    "unidad_no_micromolar": 1665,
    "otra_medida": 3134,
    "quimica_no_representada": 7457,
    "concentracion_no_puntual_positiva": 906,
    "terminal_no_representado": 1056
  },
  "aggregation": "Mediana de mediciones puntuales po

## 2. Modelo del proceso
La representación se selecciona dentro del entrenamiento. La reserva es interna, agrupada por similitud; no equivale a validación externa ni evita toda dependencia por publicaciones.

In [3]:
print(pd.read_csv(ROOT / 'results/models/validacion_reservada.csv').to_string(index=False))
print(json.dumps(read_json('results/models/protocolo.json'), ensure_ascii=False, indent=2))


objective representation  n_features  n_entities  n_train  n_holdout  n_groups  mae_log10  r2_log10  mae_baseline_log10  shared_article_ids_train_holdout  shared_publications_train_holdout
      MIC        ordered          42        1473     1160        313       818    0.37114  0.262163            0.433486                                 0                                133
     HC50        ordered          42         199      167         32       112    0.41843  0.059036            0.462394                                 0                                  9
{
  "seed": 20261017,
  "outer": "GroupShuffleSplit 20% de grupos, una particion reservada",
  "inner": "GroupKFold 5 pliegues solo en entrenamiento; menor MAE elige representacion",
  "grouping": "Componentes transitivas, igual longitud, Hamming <=2; no elimina toda homologia ni separa por publicacion",
  "scope": "Prueba interna reservada de la misma cohorte; no validacion biologica de variantes ni validacion externa",
  "publi

![Validación interna](../results/figures/validacion_predictores.png)

## 3. Sensibilidad y comparación
NSGA-II tiene cruce uniforme, mutación categórica y reparación. Se cuentan secuencias únicas. El ajuste de parámetros y las corridas finales usan semillas diferentes.

In [4]:
s = pd.read_csv(ROOT / 'results/optimization/sensibilidad.csv')
print(s.groupby(['sensitivity_parameter','sensitivity_value']).agg(HV_media=('hv_percent','mean'), HV_DE=('hv_percent','std'), segundos=('seconds','mean')).to_string())
final = pd.read_csv(ROOT / 'results/optimization/comparacion_final.csv')
print(final[['method','seed','unique_evaluations','hv_percent','seconds']].to_string(index=False))
print(json.dumps(read_json('results/optimization/estadisticas.json'), ensure_ascii=False, indent=2))


                                          HV_media     HV_DE  segundos
sensitivity_parameter sensitivity_value                               
crossover             0.500000           99.538706  0.257344  1.196866
                      0.700000           99.692817  0.188102  1.192961
                      0.900000           99.254184  0.833819  1.160451
mutation              0.023810           99.254184  0.833819  1.160451
                      0.047619           98.319889  1.883927  1.052425
                      0.095238           98.416606  1.652110  1.032545
population            30.000000          97.839704  2.914075  2.110661
                      60.000000          98.319889  1.883927  1.052425
                      100.000000         97.692569  2.091622  0.841045
   method  seed  unique_evaluations  hv_percent  seconds
  NSGA-II   401                2100   99.320418 1.195510
Aleatoria   401                2100   85.589694 0.701996
Aleatoria   402                2100   87.107488 

![Sensibilidad](../results/figures/sensibilidad.png)

![Comparación](../results/figures/comparacion_calidad_tiempo.png)

## 4. Convergencia
El promedio y el mejor valor se registran por objetivo en cada población superviviente. Los extremos pueden corresponder a individuos distintos. El hipervolumen usa todo el archivo acumulado.

In [5]:
h = pd.read_csv(ROOT / 'results/optimization/historia_comparacion.csv')
example = h[(h.method == 'NSGA-II') & (h.seed == 401)]
print(example[['generation_or_batch','n_evaluations','mean_log10_MIC','best_log10_MIC','mean_log10_HC50','best_log10_HC50','hv_percent']].to_string(index=False))


 generation_or_batch  n_evaluations  mean_log10_MIC  best_log10_MIC  mean_log10_HC50  best_log10_HC50  hv_percent
                   1             60        0.938291        0.786654         1.898559         2.227866   71.018651
                   2            120        0.875682        0.722069         2.002086         2.251523   78.178008
                   3            180        0.855323        0.716074         2.075196         2.283330   80.654608
                   4            240        0.826862        0.680695         2.106269         2.283330   82.799664
                   5            300        0.819659        0.680695         2.151648         2.283330   82.799664
                   6            360        0.789566        0.676166         2.183033         2.288480   83.704217
                   7            420        0.761306        0.671149         2.205254         2.361765   88.038305
                   8            480        0.750822        0.669447         2.214086    

![Aptitud por generación](../results/figures/aptitud_promedio_y_mejor.png)

![Hipervolumen](../results/figures/convergencia_hipervolumen.png)

## 5. Candidatos y comprobaciones
La procedencia indica en qué semillas finales NSGA-II evaluó cada candidato. El cociente HC50/MIC sigue siendo una razón de predicciones.

In [6]:
c = pd.read_csv(ROOT / 'results/optimization/candidatos_encontrados.csv')
print(c[['criterio','sequence','mutations_1based','pred_MIC_uM','pred_HC50_uM','seeds_found']].to_string(index=False))
print(json.dumps(read_json('results/auditoria_resultados.json'), ensure_ascii=False, indent=2))


               criterio              sequence mutations_1based  pred_MIC_uM  pred_HC50_uM                     seeds_found
     Menor MIC predicha GRMKYIKPLIPHAVKAIKKAI         F2R;L20A     4.096276     99.671973                     402,404,408
    Mayor HC50 predicha GRMKYIKPLIPHAVKAKKKLI         F2R;I17K     6.388168    299.024863 401,402,403,404,405,407,409,410
Mayor cociente predicho GRMKRIKPLIPHAVKAIKKLI          F2R;Y5R     4.187451    233.702002     402,403,404,405,408,409,410
{
  "raw_files_verified": 4355,
  "reserved_partitions_and_metrics": "verified",
  "exhaustive_domain_and_hv": "verified",
  "unique_runs_verified": 55,
  "final_pairs_initial_population": "identical_in_each_pair",
  "candidate_provenance": "observed_in_final_nsga_runs",
  "repeated_real_run": {
    "seed": 401,
    "exact_sequence_order": true,
    "prediction_atol": 1e-12,
    "note": "Se excluye el tiempo del requisito de igualdad."
  }
}


![Frentes](../results/figures/frentes_pareto.png)

La interpretación y bibliografía verificable se encuentran en [el informe](../docs/INFORME_RESULTADOS.md). No se han realizado ensayos biológicos ni una nueva ejecución de ColabFold.